# Figure SS01 (supplementary): amino-acid × 3Di entropy density of GMSC smORFs

**Status:** draft, Gadi. The ID is provisional.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figSS01_density_aa_3di.tsv.gz` | `density_aa_vs_3di_counts.npz` (400 × 400 bins of 0.011 bits over 0–4.4 bits, every sequence) | `density_aa_vs_3di.py` (job 181044341) |

- **Run:** `linsalrob/genome_entropy` `Validation/smorfs/` (#105), with the write-up at `agent-handoffs` `projects/genome_entropy_logan/smorfs/manuscript.md`.
  - Software: `genome_entropy` 0.3.0, the ModernProst `gbouras13/modernprost-50M` encoder at revision `2d68d506`, encoding size 50,000, and translation table 11.
  - Hardware: V100 (`gpuvolta`).
- **Results location:** Gadi `/g/data/ob80/re3494/Projects/smORFs/results/`, mirrored on Teams at `genome_entropy:General/smORFs/results/`.
- **Reduction:** `manuscript/prep/smorfs_prepare_data.py`. It produces histograms over the per-smORF table (PBS job 181367025, 0.06 SU) and copies the AUC, model and stripe tables verbatim.
- **Data:** the GMSC benchmark sets as supplied by the GMSC authors.
  - **Positives:** 33,647,794 high-quality smORFs with RNAcode plus transcriptomic or proteomic support, or homology to known small proteins.
  - **Negatives:** 10,079,115 conserved, in-frame GMSC sequences that failed coding tests.
  - Lengths are 10–99 aa, with no 90 aa floor. The negatives are not random ORFs.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}

import math
from matplotlib.colors import LogNorm, TwoSlopeNorm
CLS = {"positive": "#0072B2", "negative": "#D55E00"}


In [ ]:
D = pd.read_csv(DATA / "figSS01_density_aa_3di.tsv.gz", sep="\t")
edges = np.linspace(0, 4.4, 401)
H = {}
for c in ("positive", "negative"):
    g = D[D.dataset == c]
    h = np.zeros((400, 400))
    h[g.i_aa, g.j_3di] = g.n  # h[amino acid, 3Di]
    H[c] = h / h.sum()
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.6), gridspec_kw={"wspace": 0.42})
norm = LogNorm(1e-7, 1e-3)
for ax, c, letter in zip(axes[:2], ("positive", "negative"), "ab"):
    m = ax.pcolormesh(edges, edges, np.where(H[c] > 0, H[c], np.nan).T, norm=norm,
                      cmap="viridis", rasterized=True)
    ax.set_title(f"{c}", loc="left", color=CLS[c], fontsize=7)
    panel(ax, letter)
cb = fig.colorbar(m, ax=axes[1], fraction=0.05, pad=0.04)
cb.ax.set_title("fraction\nper bin", fontsize=5.5, loc="left")
cb.ax.tick_params(labelsize=5.5)
diff = H["positive"] - H["negative"]
lim = np.nanmax(np.abs(diff))
m2 = axes[2].pcolormesh(edges, edges, np.where((H["positive"] + H["negative"]) > 0, diff, np.nan).T,
                        norm=TwoSlopeNorm(0, -lim, lim), cmap="RdBu", rasterized=True)
axes[2].set_title("positive − negative", loc="left", fontsize=7)
panel(axes[2], "c")
cb2 = fig.colorbar(m2, ax=axes[2], fraction=0.05, pad=0.03)
cb2.ax.tick_params(labelsize=5.5)
cb2.ax.set_title("Δ fraction", fontsize=5.5, loc="left")
for k, ax in enumerate(axes):
    ax.set_xlim(2.4, 4.4)
    ax.set_ylim(0, 4.2)
    ax.set_xlabel("amino-acid entropy (bits)")
    if k == 0:
        ax.set_ylabel("3Di entropy (bits)")
save(fig, "figSS01_smorf_density")


In [ ]:
print({c: int(D[D.dataset == c].n.sum()) for c in ("positive", "negative")})


## Caption (draft)

**Figure SS01. Amino-acid against 3Di entropy for all 43,726,909 GMSC smORFs.**

Bins are 0.011 bits on each axis. **(a, b)** The fraction of each class per bin, on a log scale. **(c)** The positive fraction minus the negative fraction per bin. Blue marks bins enriched in positives, red bins enriched in negatives.

- **Enrichment for positives:** at high 3Di entropy (above about 2.3 bits) when amino-acid entropy is above about 3.6 bits.
- **Enrichment for negatives:** in a band at 1.0–1.6 bits of 3Di entropy centred on amino-acid entropy ≈ 4.0.

The class difference is therefore a joint function of the two axes, and a threshold on either axis alone cannot capture it.
